In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ==========================================
# ЗАДАНИЕ: Двухслойная RNN для классификации текстов
# ==========================================

class StackedRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes, num_layers=2):
        super().__init__()

        # TODO 1: Инициализация слоев
        # 1. Создай self.embedding (vocab_size -> embed_dim, padding_idx=0)
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim, padding_idx=0)

        # 2. Создай self.rnn (вход embed_dim, память hidden_size).
        #    ОБЯЗАТЕЛЬНО укажи аргументы: num_layers=num_layers и batch_first=True
        self.rnn = nn.RNN(input_size=embed_dim, hidden_size=hidden_size,  num_layers=num_layers,batch_first=True)

        # 3. Создай self.fc (hidden_size -> num_classes)

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x имеет размер [batch_size, seq_len]

        # TODO 2: Прямой проход (Следи за размерностями!)
        # 1. Пропусти x через embedding. Результат: [batch_size, seq_len, embed_dim]
        emb = self.embedding(x)

        # 2. Пропусти результат через self.rnn.
        #    Напоминание: rnn возвращает кортеж (output, hidden)
        out, hidden = self.rnn(emb)

        # 3. Извлеки память ВЕРХНЕГО слоя.
        #    В многослойной RNN тензор hidden имеет размер: [num_layers, batch_size, hidden_size].
        #    Чтобы получить память только последнего слоя, нужно сделать срез: взять последний элемент (-1) по нулевой размерности.
        top_layer_hidden = hidden[-1]

        # 4. Передай top_layer_hidden в классификатор (self.fc) и верни логиты.
        logits = self.fc(top_layer_hidden)
        return logits

In [ ]:
# ==========================================
# ЗАДАНИЕ 3: Проверка модели
# ==========================================
# TODO 3:
# 1. Инициализируй модель
rnn = StackedRNNClassifier(vocab_size=5000, embed_dim=100, hidden_size=128, num_classes=4, num_layers=2)

# 2. Создай фейковый батч данных: 8 предложений по 15 слов в каждом.
fake_batch = torch.randint(1, 5000, (8, 15))

# 3. Сделай forward pass: пропусти этот батч через модель.
logits = rnn(fake_batch)

# 4. Распечатай размерность (shape) выхода. Если всё написано верно, должно получиться [8, 4].
print(logits.shape)

torch.Size([8, 4])
